# AI vs. Classic IR: BM25 vs. Neural Dense Retrieval on News (Problem Statement 1: Vocabulary Mismatch)

**Corpus:** HuffPost News Category Dataset v3 (headline + short description).
**Engines:** A = BM25 from scratch, B = dense retrieval (SBERT + FAISS, cosine), C = hybrid RRF (extra).
**Query suite (20):** 4 negation, 4 homonym, 4 temporal, 5 vocabulary-mismatch, 3 lexical-anchor.
**Ground truth:** rule-based labels over the *whole corpus* (regex concept lexicons + date windows). It does not use either engine, so neither engine can bias its own judgments.

Evaluation (NDCG@5/10, MAP@100, Recall@100/1000, hard-negative rate, time & memory, coverage-bucket analysis, complementarity) is in the last two cells. Run on a GPU runtime; set `CACHE_DIR` to a Google Drive folder so the embeddings survive resets.

In [ ]:
!pip install -q sentence-transformers faiss-cpu nltk

import os
import re
import math
import time
import glob
import hashlib
import unicodedata
from functools import lru_cache
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import torch
import faiss
from sentence_transformers import SentenceTransformer
from nltk.stem import PorterStemmer

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_columns", 20)


# Experiment configuration: change in values here only.

MODEL_NAME = "sentence-transformers/multi-qa-MiniLM-L6-cos-v1"  # retrieval-tuned, 384-d, same speed as all-MiniLM-L6-v2
QUERY_PREFIX = ""     # e.g. "query: " for E5 models
DOC_PREFIX = ""       # e.g. "passage: " for E5 models
USE_STEMMING = True   # False = BM25 on exact surface forms only (a cleaner "pure vocabulary mismatch" baseline)
BM25_K1, BM25_B = 1.5, 0.75
RRF_K, RRF_DEPTH = 60, 500
CACHE_DIR = "."      # e.g. "/content/drive/MyDrive/ir_cache" (after drive.mount) so cached embeddings survive Colab resets

print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()} | model: {MODEL_NAME} | stemming: {USE_STEMMING}")

In [ ]:
REQUIRED_COLS = ["headline", "short_description", "category", "date"]
CANDIDATE_PATHS = (
    ["News_Category_Dataset_v3.json.zip", "News_Category_Dataset_v3.json", "news_corpus.csv"]
    + sorted(glob.glob("/kaggle/input/**/News_Category_Dataset_v3.json*", recursive=True))
    + sorted(glob.glob("/content/**/News_Category_Dataset_v3.json*", recursive=True))
)
loaded_path = next((p for p in CANDIDATE_PATHS if os.path.exists(p)), None)
if loaded_path is None:
    try:
        from google.colab import files
        print("Please upload News_Category_Dataset_v3.json.zip (or .json):")
        uploaded = files.upload()
        loaded_path = next(iter(uploaded))
    except ImportError:
        raise FileNotFoundError("Place News_Category_Dataset_v3.json(.zip) in the working directory.")

if loaded_path.endswith(".csv"):
    df = pd.read_csv(loaded_path)
else:
    df = pd.read_json(loaded_path, lines=True)

missing_cols = [c for c in REQUIRED_COLS if c not in df.columns]
if missing_cols:
    raise ValueError(f"Dataset is missing required columns: {missing_cols}")

df = df.reset_index(drop=True)
if "doc_id" not in df.columns:               # works for both the JSON and the CSV branch
    df["doc_id"] = [f"N{i:06d}" for i in range(len(df))]
assert df["doc_id"].is_unique, "doc_id must be unique"

df["headline"] = df["headline"].fillna("").astype(str)
df["short_description"] = df["short_description"].fillna("").astype(str)
df["category"] = df["category"].fillna("UNKNOWN").astype(str)
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["search_text"] = (df["headline"].str.strip() + ". " + df["short_description"].str.strip()).str.strip()

doc_id_to_idx = {doc_id: idx for idx, doc_id in enumerate(df["doc_id"])}
idx_to_doc_id = df["doc_id"].to_numpy()

print(f"Corpus loaded from '{loaded_path}': {len(df):,} documents.")
print(f"Date range: {df['date'].min().date()} -> {df['date'].max().date()} | unparseable dates: {int(df['date'].isna().sum())}")
print("Docs per year (check that temporal query windows contain enough documents):")
print(df["date"].dt.year.value_counts().sort_index().to_string())
display(df[["doc_id", "headline", "short_description", "category", "date"]].head(5))

In [ ]:

# Text processing for BM25 (Engine A)
# Design choices (document these in the report):
#   * standard English stopword list, but NEGATION words are kept so negation
#     queries are not silently erased before retrieval;
#   * hyphenated words are split ("bird-watcher" == "bird watcher");
#   * "U.S." -> "us", contractions are expanded ("don't" -> "do not"),
#     possessives dropped, accents stripped;
#   * optional Porter stemming (USE_STEMMING in the config cell).

_BASE_STOPWORDS = """
i me my myself we our ours ourselves you your yours yourself yourselves he him his himself she her hers herself
it its itself they them their theirs themselves what which who whom this that these those am is are was were be been being
have has had having do does did doing a an the and but if or because as until while of at by for with about against between
into through during before after above below to from up down in out on off over under again further then once here there when
where why how all any both each few more most other some such only own same so than too very s t can will just don should now
said says would could also may might must shall
""".split()
NEGATION_WORDS = frozenset({"no", "nor", "not", "never", "none", "nothing", "nobody", "nowhere", "neither", "without", "cannot"})
STOPWORDS = frozenset(_BASE_STOPWORDS) - NEGATION_WORDS

TOKEN_PATTERN = re.compile(r"[a-z0-9]+")
_STEMMER = PorterStemmer()


def strip_accents(text):
    return "".join(c for c in unicodedata.normalize("NFKD", text) if not unicodedata.combining(c))


@lru_cache(maxsize=None)
def _stem(token):
    if USE_STEMMING and token.isalpha() and len(token) > 3:
        return _STEMMER.stem(token)
    return token


def normalize_text(text):
    t = strip_accents(text.lower())
    t = t.replace("’", "'").replace("‘", "'").replace("`", "'")
    t = re.sub(r"\bu\.s\.(?:a\.?)?", " us ", t)
    t = re.sub(r"\bu\.k\.", " uk ", t)
    t = re.sub(r"\bcan't\b", "can not", t)
    t = re.sub(r"\bwon't\b", "will not", t)
    t = re.sub(r"\bcannot\b", "can not", t)
    t = re.sub(r"n't\b", " not", t)
    t = re.sub(r"'(?:s|re|ve|ll|d|m)\b", "", t)
    t = t.replace("'", "")
    t = re.sub(r"[-–—/_]", " ", t)
    return t


def custom_tokenize(text, remove_stopwords=True, stem=True):
    tokens = []
    for tok in TOKEN_PATTERN.findall(normalize_text(text)):
        if remove_stopwords and tok in STOPWORDS:
            continue
        if len(tok) < 2 and not tok.isdigit():
            continue
        tokens.append(_stem(tok) if stem else tok)
    return tokens


#  unit tests for the tokenizer
assert custom_tokenize("Bird-Watcher") == custom_tokenize("bird watcher")
assert "us" in custom_tokenize("U.S. attorney's office")
assert "not" in custom_tokenize("movies where the main character does not die")
assert custom_tokenize("don't") == ["not"]
assert len(custom_tokenize("without meat")) == 2
assert custom_tokenize("Erdoğan") == custom_tokenize("erdogan")
if USE_STEMMING:
    assert custom_tokenize("Airlines recall vehicles") == custom_tokenize("airline recalls vehicle")
print(f"Tokenizer OK | stopwords: {len(STOPWORDS)} | negation words kept: {sorted(NEGATION_WORDS)}")
print("Example:", custom_tokenize("Woman Who Called Cops On Black Bird-Watcher Loses Lawsuit Against Ex-Employer"))

In [ ]:
class ScratchBM25Engine:
    """Okapi BM25 over an inverted index, implemented from scratch (Manning et al., Ch. 6/11)."""

    def __init__(self, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.N = 0
        self.avgdl = 0.0
        self.doc_len = None
        self.df_table = {}
        self.idf_table = {}
        self.inverted_index = {}

    def fit(self, corpus_texts):
        t0 = time.time()
        self.N = len(corpus_texts)
        self.doc_len = np.zeros(self.N, dtype=np.int32)
        temp_postings = defaultdict(lambda: ([], []))
        for doc_idx, text in enumerate(corpus_texts):
            tokens = custom_tokenize(text)
            self.doc_len[doc_idx] = len(tokens)
            for term, tf in Counter(tokens).items():
                doc_list, tf_list = temp_postings[term]
                doc_list.append(doc_idx)
                tf_list.append(tf)
        self.avgdl = float(np.mean(self.doc_len))
        for term, (doc_list, tf_list) in temp_postings.items():
            df_t = len(doc_list)
            self.df_table[term] = df_t
            self.idf_table[term] = math.log(1.0 + (self.N - df_t + 0.5) / (df_t + 0.5))
            self.inverted_index[term] = (np.array(doc_list, dtype=np.int32), np.array(tf_list, dtype=np.float32))
        self.fit_seconds = time.time() - t0
        print(
            f"[Engine A: BM25] Indexed {self.N:,} docs | Vocab: {len(self.inverted_index):,} terms | "
            f"Avg doc length: {self.avgdl:.2f} tokens | Time: {time.time() - t0:.2f}s"
        )
        return self

    def score_all(self, query_text):
        scores = np.zeros(self.N, dtype=np.float32)
        for term, q_tf in Counter(custom_tokenize(query_text)).items():
            if term not in self.inverted_index:
                continue
            doc_indices, doc_tfs = self.inverted_index[term]
            norm = self.k1 * (1.0 - self.b + self.b * (self.doc_len[doc_indices] / self.avgdl))
            scores[doc_indices] += self.idf_table[term] * ((doc_tfs * (self.k1 + 1.0)) / (doc_tfs + norm)) * q_tf
        return scores

    def search(self, query_text, top_k=100):
        """Return only documents with a strictly positive score (documents sharing no query term are NOT ranked)."""
        scores = self.score_all(query_text)
        positive = np.flatnonzero(scores > 0)
        if len(positive) == 0:
            return np.array([], dtype=np.int64), np.array([], dtype=np.float32)
        if len(positive) > top_k:
            part = np.argpartition(scores[positive], -top_k)[-top_k:]
            positive = positive[part]
        order = np.lexsort((positive, -scores[positive]))   # score desc, then doc index asc (deterministic ties)
        top_indices = positive[order]
        return top_indices, scores[top_indices]


bm25_engine = ScratchBM25Engine(k1=BM25_K1, b=BM25_B).fit(df["search_text"].tolist())

#  smoke tests
assert len(bm25_engine.search("zzqxv qqzvz")[0]) == 0, "a query with no matching term must return nothing"
_idx, _sc = bm25_engine.search("hurricane", top_k=5)
assert 1 <= len(_idx) <= 5 and (_sc > 0).all() and (np.diff(_sc) <= 0).all()
print("BM25 smoke tests passed.")

In [ ]:
def corpus_fingerprint(texts):
    h = hashlib.md5()
    for t in texts:
        h.update(t.encode("utf-8", "ignore"))
        h.update(b"\x00")
    return h.hexdigest()[:10]


device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    print("WARNING: no GPU found. Encoding ~210k articles on CPU is very slow. "
          "Use Runtime > Change runtime type > T4 GPU (do this BEFORE uploading the dataset; switching restarts the VM).")
dense_encoder = SentenceTransformer(MODEL_NAME, device=device)
embedding_dim = (dense_encoder.get_embedding_dimension() if hasattr(dense_encoder, "get_embedding_dimension")
                 else dense_encoder.get_sentence_embedding_dimension())

doc_texts = [DOC_PREFIX + t for t in df["search_text"]]
# The cache key contains model name, corpus size and a content hash, so a changed model or corpus can never load stale vectors.
os.makedirs(CACHE_DIR, exist_ok=True)
EMBEDDING_CACHE_PATH = os.path.join(CACHE_DIR, f"emb_{MODEL_NAME.split('/')[-1]}_{len(df)}_{corpus_fingerprint(doc_texts)}.npy")

corpus_embeddings = None
dense_encode_seconds = None   # stays None when embeddings come from the cache
if os.path.exists(EMBEDDING_CACHE_PATH):
    cached = np.load(EMBEDDING_CACHE_PATH)
    if cached.shape == (len(df), embedding_dim):
        corpus_embeddings = cached.astype(np.float32)
        print(f"[Engine B: Dense] Loaded cached embeddings: {EMBEDDING_CACHE_PATH}")
    else:
        print(f"[Engine B: Dense] Cache shape {cached.shape} does not match corpus; re-encoding.")

if corpus_embeddings is None:
    print(f"[Engine B: Dense] Encoding {len(df):,} articles on {device.upper()}...")
    t0 = time.time()
    corpus_embeddings = dense_encoder.encode(
        doc_texts,
        batch_size=512 if device == "cuda" else 128,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)
    np.save(EMBEDDING_CACHE_PATH, corpus_embeddings)
    dense_encode_seconds = time.time() - t0
    print(f"[Engine B: Dense] Encoded in {dense_encode_seconds:.1f}s, cached to {EMBEDDING_CACHE_PATH}")

faiss_index = faiss.IndexFlatIP(embedding_dim)   # inner product on unit vectors == cosine similarity
faiss_index.add(corpus_embeddings)
print(f"[Engine B: FAISS] Index built with {faiss_index.ntotal:,} vectors (dim={embedding_dim}).")


class DenseFAISSEngine:
    def __init__(self, encoder, index, query_prefix=""):
        self.encoder = encoder
        self.index = index
        self.query_prefix = query_prefix

    def encode_query(self, query_text):
        return self.encoder.encode(
            [self.query_prefix + query_text], convert_to_numpy=True, normalize_embeddings=True
        ).astype(np.float32)[0]

    def search(self, query_text, top_k=100):
        q_emb = self.encode_query(query_text)[None, :]
        scores, indices = self.index.search(q_emb, top_k)
        return indices[0], scores[0]


dense_engine = DenseFAISSEngine(dense_encoder, faiss_index, query_prefix=QUERY_PREFIX)

In [ ]:
class HybridRRFEngine:
    """Reciprocal Rank Fusion of BM25 and dense rankings (extra engine, used as a reference point)."""

    def __init__(self, bm25_eng, dense_eng, rrf_k=60, candidate_depth=500):
        self.bm25 = bm25_eng
        self.dense = dense_eng
        self.rrf_k = rrf_k
        self.candidate_depth = candidate_depth

    def search(self, query_text, top_k=100):
        bm25_idx, _ = self.bm25.search(query_text, top_k=self.candidate_depth)    # positive-score docs only
        dense_idx, _ = self.dense.search(query_text, top_k=self.candidate_depth)
        rrf_scores = defaultdict(float)
        for rank_0, doc_idx in enumerate(bm25_idx):
            rrf_scores[int(doc_idx)] += 1.0 / (self.rrf_k + rank_0 + 1)
        for rank_0, doc_idx in enumerate(dense_idx):
            rrf_scores[int(doc_idx)] += 1.0 / (self.rrf_k + rank_0 + 1)
        ranked = sorted(rrf_scores.items(), key=lambda kv: (-kv[1], kv[0]))[:top_k]
        top_indices = np.array([d for d, _ in ranked], dtype=np.int64)
        top_scores = np.array([s for _, s in ranked], dtype=np.float32)
        return top_indices, top_scores


hybrid_engine = HybridRRFEngine(bm25_engine, dense_engine, rrf_k=RRF_K, candidate_depth=RRF_DEPTH)
print(f"[Engine C: Hybrid RRF] Ready (rrf_k={RRF_K}, pool depth={RRF_DEPTH}).")

In [ ]:

# Query suite: 20 queries = 4 negation + 4 homonym + 4 temporal + 5 vocabulary-mismatch + 3 lexical-anchor.

# Every query carries an explicit, machine-checkable RELEVANCE RULE that is written from the *concept*
# the user means, not from the words in the query and not from either retrieval engine:
#   topic  : OR-list of regexes. A document must match one to be relevant at all.
#   must   : list of OR-lists. A topic match that satisfies ALL of them gets grade 2; otherwise grade 1.
#   forbid : OR-list. Documents matching it are NOT relevant (negation / wrong homonym sense).
#   date_from / date_to : relevance window (temporal queries). Outside the window => not relevant.
#   Documents that match the topic but violate forbid/date are recorded as HARD NEGATIVES:
#   exactly the documents a keyword or semantic engine is tempted to return.
# Regexes are matched (word-bounded, case-insensitive) against "headline. short_description".


COVID = [r"covid(?:-19)?", r"coronavirus", r"sars-cov-2"]
VAX = [r"vaccin\w*", r"vax\w*", r"jabs?", r"pfizer", r"moderna", r"immuniz\w*", r"inoculat\w*"]
BOOSTER = [r"boosters?", r"third dose", r"fourth dose", r"bivalent", r"omicron-targeted", r"additional dose"]

SPORT_CTX = [r"test match\w*", r"innings", r"wickets?", r"world cup", r"ipl", r"batsm[ae]n", r"batting",
             r"bowlers?", r"bowling", r"t20", r"bcci", r"icc", r"ashes", r"stumps", r"sachin", r"tendulkar", r"kohli",
             r"dhoni", r"cricketers?", r"cricket team", r"cricket club", r"cricket board", r"cricket pitch", r"cricket bat"]
INSECT_CTX = [r"chirp\w*", r"insects?", r"bugs?", r"flour", r"protein", r"edible", r"entomophag\w*", r"larva\w*",
              r"roach\w*", r"grasshoppers?", r"locusts?", r"pests?", r"critters?", r"eating crickets?", r"cricket farm\w*",
              r"energy bars?", r"powder"]

FOOD_CTX = [r"thanksgiving", r"roast\w*", r"brin\w*", r"stuffing", r"gravy", r"leftovers?", r"oven", r"carv\w*",
            r"recipes?", r"bacon", r"sandwich\w*", r"burgers?", r"meatballs?", r"chili", r"bird", r"fry\w*",
            r"fried", r"breast", r"butterball", r"wings?", r"drumsticks?", r"dressing", r"cranberry", r"feast"]
FOOD_STRICT = [r"thanksgiving", r"roast\w*", r"brin\w*", r"stuffing", r"gravy", r"leftovers?", r"oven", r"carv\w*",
               r"recipes?", r"butterball", r"drumsticks?", r"cranberry", r"turkey bacon", r"turkey sandwich\w*",
               r"turkey burgers?", r"ground turkey", r"turkey breast", r"turkey legs?", r"turkey day"]
COUNTRY_CTX = [r"erdogan", r"ankara", r"istanbul", r"turkish", r"syria\w*", r"kurd\w*", r"isis", r"coup", r"nato",
               r"armenian\w*", r"bosphorus", r"aegean", r"antalya", r"cappadocia", r"izmir", r"bodrum", r"ottoman", r"greece"]

QUERY_SPECS = [
    # ---------------- NEGATION (4) ----------------
    dict(query_id="Q01", family="Negation", subtype="Excluded sub-topic",
         query="COVID vaccine news that is not about booster shots",
         topic=COVID, must=[VAX], forbid=BOOSTER,
         hypothesis="Both engines rank booster articles high: negation is invisible to term matching and weak in embeddings."),
    dict(query_id="Q02", family="Negation", subtype="Excluded entity",
         query="Republican presidential primary coverage that does not mention Donald Trump",
         topic=[r"(?<!santa )(?<!penelope )cruz", r"rubio", r"kasich", r"ben carson", r"jeb bush", r"carly fiorina",
                r"chris christie", r"mike huckabee", r"rick santorum", r"rand paul", r"lindsey graham"],
         must=[[r"primary", r"primaries", r"caucus\w*", r"campaign\w*", r"debates?", r"candidates?", r"gop", r"republicans?", r"nomination"]],
         forbid=[r"trump\w*"], date_from="2015-06-01", date_to="2016-07-31",
         hypothesis="Both engines retrieve Trump-centred articles because 'Trump' is the strongest term/concept in the query."),
    dict(query_id="Q03", family="Negation", subtype="Excluded event",
         query="hurricane coverage that has nothing to do with Superstorm Sandy",
         topic=[r"hurricanes?"], forbid=[r"sandy", r"superstorm"],
         hypothesis="BM25 and dense both over-retrieve the Sandy articles named in the query."),
    dict(query_id="Q04", family="Negation", subtype="Absent attribute",
         query="dinner recipes that do not contain any meat",
         topic=[r"vegetarian", r"vegan", r"meatless", r"meat-free", r"plant-based", r"veggie"],
         must=[[r"recipes?", r"dinners?", r"dishes?", r"meals?", r"cook\w*", r"skillet", r"soup", r"pasta", r"salad", r"curry", r"stew"]],
         forbid=[r"chicken", r"beef", r"pork", r"bacon", r"steak", r"sausages?", r"lamb", r"ham", r"turkey", r"meatballs?",
                 r"prosciutto", r"pepperoni", r"salmon", r"shrimp", r"fish"],
         hypothesis="BM25 matches the literal word 'meat'; dense may capture 'vegetarian' semantics better."),

    # ---------------- HOMONYMS (4) ----------------
    dict(query_id="Q05", family="Homonym", subtype="Cricket = sport",
         query="cricket match highlights",
         topic=[r"cricket", r"cricketers?"], must=[SPORT_CTX], forbid=INSECT_CTX,
         hypothesis="BM25 returns insect articles too; dense should separate the sense from 'match highlights'."),
    dict(query_id="Q06", family="Homonym", subtype="Cricket = insect",
         query="cricket insect life cycle",
         topic=[r"cricket", r"crickets"], must=[INSECT_CTX], forbid=SPORT_CTX,
         hypothesis="Few relevant documents exist; BM25 and dense both risk surfacing sport articles."),
    dict(query_id="Q07", family="Homonym", subtype="Turkey = food",
         query="how to cook a juicy turkey for dinner",
         topic=[r"turkeys?"], must=[FOOD_CTX], forbid=COUNTRY_CTX,
         hypothesis="Context words ('cook', 'dinner') should help both engines; dense expected to be cleaner."),
    dict(query_id="Q08", family="Homonym", subtype="Turkey = country",
         query="Turkey's government crackdown after the failed coup attempt",
         topic=[r"turkey"], must=[[p for p in COUNTRY_CTX if p != r"coup"]], forbid=FOOD_STRICT,
         hypothesis="BM25 relies on 'coup'/'crackdown'; dense should place the political sense of Turkey correctly."),

    # ---------------- TEMPORAL (4) ----------------
    dict(query_id="Q09", family="Temporal", subtype="Event window (explicit date)",
         query="reactions to the presidential election results in the week after November 8, 2016",
         topic=[r"trump", r"clinton", r"hillary", r"president-elect", r"electoral college"],
         must=[[r"won", r"wins?", r"victory", r"defeat\w*", r"lost", r"loss", r"concede\w*", r"votes?", r"voters?"]],
         date_from="2016-11-08", date_to="2016-11-15",
         hypothesis="Neither engine reads dates: both retrieve election articles from any month."),
    dict(query_id="Q10", family="Temporal", subtype="Event window (season + year)",
         query="Rio de Janeiro Summer Olympics 2016 coverage",
         topic=[r"olympics?", r"olympians?", r"olympic", r"rio 2016", r"gold medals?", r"rio games"],
         date_from="2016-07-20", date_to="2016-09-05",
         hypothesis="Olympic articles from 2012/2014/2018 are hard negatives for both engines."),
    dict(query_id="Q11", family="Temporal", subtype="Relative time ('right after')",
         query="gun control debate in the weeks right after the Parkland school shooting",
         topic=[r"gun control", r"guns?", r"firearms?", r"nra", r"parkland", r"second amendment", r"assault weapons?", r"ar-15",
                r"school shootings?", r"march for our lives", r"gun violence", r"gun laws?"],
         date_from="2018-02-14", date_to="2018-04-14",
         hypothesis="Earlier (Sandy Hook, Orlando, Las Vegas) and later gun-debate articles pollute both rankings."),
    dict(query_id="Q12", family="Temporal", subtype="Relative time ('before')",
         query="early news about the new coronavirus outbreak, before it was declared a pandemic",
         topic=[r"coronavirus", r"covid(?:-19)?", r"wuhan", r"novel virus", r"2019-ncov"],
         date_from="2020-01-01", date_to="2020-03-10",
         hypothesis="Later pandemic coverage dominates both rankings; 'early' and 'before' are ignored."),

    # ---------------- VOCABULARY MISMATCH (5) ----------------
    dict(query_id="Q13", family="Vocabulary Mismatch", subtype="Synonym / paraphrase",
         query="doctors say lack of rest is hurting people's daytime alertness",
         topic=[r"sleep", r"sleeping", r"sleepless", r"sleeplessness", r"sleep-deprived", r"insomnia", r"drowsy", r"fatigue", r"circadian"],
         must=[[r"health", r"study", r"studies", r"research\w*", r"experts?", r"risks?", r"scientists?", r"brain", r"heart", r"warn\w*"]],
         hypothesis="Query shares almost no word with 'sleep' articles: dense should beat BM25."),
    dict(query_id="Q14", family="Vocabulary Mismatch", subtype="Synonym / paraphrase",
         query="carmaker pulls faulty automobiles off the road over safety fears",
         topic=[r"recalls?", r"recalled", r"recalling"],
         must=[[r"cars?", r"vehicles?", r"automakers?", r"toyota", r"ford motors?", r"general motors", r"gm", r"honda", r"chrysler",
                r"nissan", r"takata", r"airbags?", r"tesla", r"hyundai", r"volkswagen", r"vw", r"fiat", r"bmw", r"mercedes", r"trucks?",
                r"suvs?", r"ignition", r"brakes?"]],
         hypothesis="The key term 'recall' is absent from the query: dense should beat BM25."),
    dict(query_id="Q15", family="Vocabulary Mismatch", subtype="Implicit concept",
         query="everyday essentials getting pricier while paychecks stagnate",
         topic=[r"inflation", r"cost of living", r"rising prices", r"higher prices", r"prices? (?:are )?(?:rising|soaring|climbing|jumping|surging)",
                r"price hikes?", r"price increases?", r"pricier", r"more expensive", r"unaffordable", r"grocery prices", r"food prices",
                r"gas prices", r"wage stagnation", r"stagnant wages?", r"wages? stagnat\w*", r"cost of (?:food|groceries|housing|rent)"],
         hypothesis="'Inflation' never appears in the query: dense expected to win."),
    dict(query_id="Q16", family="Vocabulary Mismatch", subtype="Scientific vs colloquial",
         query="women's myocardial infarction symptoms and risk factors",
         topic=[r"heart attacks?", r"cardiac arrest", r"myocardial infarction", r"cardiac events?", r"heart disease", r"coronary", r"cardiovascular"],
         must=[[r"women", r"woman", r"female"]],
         hypothesis="News text says 'heart attack', not 'myocardial infarction': dense expected to win."),
    dict(query_id="Q17", family="Vocabulary Mismatch", subtype="Paraphrase",
         query="passenger who attacked a stewardess is barred from flying forever",
         topic=[r"flight attendants?", r"stewardess\w*", r"cabin crew", r"airline employees?", r"air hostess\w*"],
         must=[[r"punch\w*", r"hit", r"assault\w*", r"slap\w*", r"bit", r"bite", r"struck", r"hitting", r"unruly",
                r"violent", r"banned", r"ban", r"kick\w*", r"abus\w*", r"fight\w*"]],
         hypothesis="'stewardess'/'barred'/'forever' vs 'flight attendant'/'banned'/'for life': dense expected to win."),

    # ---------------- LEXICAL ANCHORS (3) ----------------
    dict(query_id="Q18", family="Lexical Anchor", subtype="Rare proper noun", min_rel=2,
         query="Vontae Davis quits during Bills game",
         topic=[r"vontae davis"],
         hypothesis="Exact rare name: BM25 expected to match or beat dense."),
    dict(query_id="Q19", family="Lexical Anchor", subtype="Exact full name", min_rel=3,
         query="Amy Cooper Central Park video",
         topic=[r"amy cooper"],
         hypothesis="Exact full name: BM25 expected to be very strong; dense may drift to similar incidents."),
    dict(query_id="Q20", family="Lexical Anchor", subtype="Exact multi-word entity",
         query="Dakota Access Pipeline protest at Standing Rock",
         topic=[r"dakota access", r"standing rock", r"dapl"],
         hypothesis="Entity name present verbatim in documents: BM25 expected to be competitive."),
]

for spec in QUERY_SPECS:
    spec.setdefault("must", [])
    spec.setdefault("forbid", [])
    spec.setdefault("date_from", None)
    spec.setdefault("date_to", None)
    spec.setdefault("min_rel", 10)

assert len(QUERY_SPECS) == 20 and len({s["query_id"] for s in QUERY_SPECS}) == 20
query_plan_df = pd.DataFrame([{k: s[k] for k in ("query_id", "family", "subtype", "query", "hypothesis")} for s in QUERY_SPECS])
print(query_plan_df["family"].value_counts().to_string())
display(query_plan_df[["query_id", "family", "subtype", "query"]])

In [ ]:

# Ground truth (qrels) from the rules in the query suite.
# Every document of the corpus is judged for every query, so there is no pooling step and no engine can influence the labels.
#   grade 2 = topic + all `must` groups, not forbidden, inside the date window
#   grade 1 = topic matches but a `must` group is missing (partially relevant / sense unresolved)
#   grade 0 = everything else. Topic matches that violate `forbid` or the date window are logged as HARD NEGATIVES.
# Use grade >= 2 for strict binary relevance, grade >= 1 for lenient.

MANUAL_QREL_OVERRIDES = {
    # "Q05": {"N012345": 0, "N054321": 2},   # <- after reading the audit sample we can force individual judgments here
}

N_DOCS = len(df)
match_text = pd.Series([strip_accents(t.lower().replace("’", "'")) for t in df["search_text"]], index=df.index)
_mask_cache = {}


def match_mask(alternatives):
    key = tuple(alternatives)
    if key not in _mask_cache:
        pattern = r"\b(?:" + "|".join(alternatives) + r")\b"
        _mask_cache[key] = match_text.str.contains(pattern, regex=True).to_numpy()
    return _mask_cache[key]


def date_mask(spec):
    mask = np.ones(N_DOCS, dtype=bool)
    if spec["date_from"]:
        mask &= (df["date"] >= pd.Timestamp(spec["date_from"])).to_numpy()
    if spec["date_to"]:
        mask &= (df["date"] <= pd.Timestamp(spec["date_to"])).to_numpy()
    return mask


neg_tokens = set(custom_tokenize(" ".join(NEGATION_WORDS)))
qrels_dict, hard_neg_dict, stats_rows = {}, {}, []
rng = np.random.RandomState(SEED)

for spec in QUERY_SPECS:
    qid = spec["query_id"]
    topic = match_mask(spec["topic"])
    must_ok = np.ones(N_DOCS, dtype=bool)
    for group in spec["must"]:
        must_ok &= match_mask(group)
    forbidden = match_mask(spec["forbid"]) if spec["forbid"] else np.zeros(N_DOCS, dtype=bool)
    in_window = date_mask(spec)

    core = topic & ~forbidden & in_window
    grades = np.zeros(N_DOCS, dtype=np.int8)
    grades[core & must_ok] = 2
    grades[core & ~must_ok] = 1
    hard_neg = topic & (forbidden | ~in_window)

    for doc_id, g in MANUAL_QREL_OVERRIDES.get(qid, {}).items():
        i = doc_id_to_idx[doc_id]
        grades[i] = g
        hard_neg[i] = (g == 0)

    qrels_dict[qid] = {int(i): int(grades[i]) for i in np.flatnonzero(grades)}
    hard_neg_dict[qid] = np.flatnonzero(hard_neg & (grades == 0))

    # Diagnostic: how many grade-2 documents share at least one (stemmed, non-negation) term with the query?
    q_terms = set(custom_tokenize(spec["query"])) - neg_tokens
    rel_idx = np.flatnonzero(grades == 2)
    sample = rng.choice(rel_idx, size=min(1000, len(rel_idx)), replace=False) if len(rel_idx) else []
    overlap = float(np.mean([bool(q_terms & set(custom_tokenize(df.at[i, "search_text"]))) for i in sample])) if len(sample) else float("nan")

    n2, n1 = int((grades == 2).sum()), int((grades == 1).sum())
    stats_rows.append({
        "query_id": qid, "family": spec["family"], "n_grade2": n2, "n_grade1": n1,
        "n_hard_negatives": int(len(hard_neg_dict[qid])),
        "rel_docs_sharing_query_term": round(overlap, 3),
        "status": "OK" if n2 >= spec["min_rel"] else f"LOW (<{spec['min_rel']} grade-2 docs) - revise rule/query",
    })

qrel_stats_df = pd.DataFrame(stats_rows)
display(qrel_stats_df)
low = qrel_stats_df[qrel_stats_df["status"] != "OK"]
if len(low):
    print("WARNING: these queries have too few relevant documents; fix their rules before evaluating:", list(low["query_id"]))

#  persist qrels (long format)
rows = []
for spec in QUERY_SPECS:
    qid = spec["query_id"]
    for i, g in qrels_dict[qid].items():
        rows.append((qid, idx_to_doc_id[i], g, "relevant"))
    for i in hard_neg_dict[qid]:
        rows.append((qid, idx_to_doc_id[int(i)], 0, "hard_negative"))
qrels_df = pd.DataFrame(rows, columns=["query_id", "doc_id", "grade", "label_type"])
qrels_df.to_csv("qrels_graded.csv", index=False)
print(f"Saved qrels_graded.csv ({len(qrels_df):,} rows)")

#  human audit sample: open qrels_audit_sample.csv and mark each row, then fix rules / overrides
audit_rows = []
for spec in QUERY_SPECS:
    qid = spec["query_id"]
    g2 = [i for i, g in qrels_dict[qid].items() if g == 2]
    g1 = [i for i, g in qrels_dict[qid].items() if g == 1]
    hn = hard_neg_dict[qid].tolist()
    for label, pool, k in (("grade2", g2, 6), ("grade1", g1, 3), ("hard_negative", hn, 4)):
        for i in (rng.choice(pool, size=min(k, len(pool)), replace=False) if pool else []):
            audit_rows.append({
                "query_id": qid, "query": spec["query"], "label": label, "doc_id": idx_to_doc_id[int(i)],
                "date": str(df.at[int(i), "date"])[:10], "headline": df.at[int(i), "headline"],
                "short_description": df.at[int(i), "short_description"][:160], "human_verdict(ok/wrong)": "",
            })
audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv("qrels_audit_sample.csv", index=False)
print(f"Saved qrels_audit_sample.csv ({len(audit_df)} rows) - spot-check it before trusting the labels.\n")

for qid in ["Q01", "Q05", "Q09", "Q13", "Q18"]:
    sub = audit_df[audit_df["query_id"] == qid]
    print(f"--- {qid}: {sub['query'].iloc[0] if len(sub) else ''}")
    for _, r in sub.groupby("label").head(3).iterrows():
        print(f"   [{r['label']:13s}] {r['date']}  {r['headline'][:95]}")

In [ ]:
# Sanity peek only (this is NOT the evaluation): top-5 of each engine for a few queries,
# annotated with the rule-based grade so you can see the labels and engines "talk to each other".
PEEK_QUERIES = ["Q03", "Q06", "Q10", "Q13", "Q19"]
engines = {"BM25": bm25_engine, "Dense": dense_engine, "Hybrid": hybrid_engine}
hard_neg_sets = {qid: set(arr.tolist()) for qid, arr in hard_neg_dict.items()}

for qid in PEEK_QUERIES:
    spec = next(s for s in QUERY_SPECS if s["query_id"] == qid)
    print(f"\n=== {qid} [{spec['family']}] {spec['query']}")
    for name, engine in engines.items():
        idx, _ = engine.search(spec["query"], top_k=5)
        for rank, i in enumerate(idx, 1):
            i = int(i)
            tag = "HARD-NEG" if i in hard_neg_sets[qid] else ""
            print(f"  {name:6s} {rank}. [grade {qrels_dict[qid].get(i, 0)}] {df.at[i, 'headline'][:85]} ({str(df.at[i, 'date'])[:10]}) {tag}")

## Evaluation
Baseline = BM25 (Engine A). Dense (Engine B) and Hybrid RRF (Engine C) are compared against it.

In [ ]:
# ------------------------------------------------------------------
# Evaluation 1/2: ranking quality, hard-negative rate, time and memory.
# Binary metrics (MAP, Recall) treat grade >= BINARY_MIN_GRADE as relevant; NDCG uses the graded labels (gain = 2^grade - 1).
# All relevant documents of the whole corpus are known (rule-based qrels), so IDCG and recall denominators are exact.
# ------------------------------------------------------------------
import matplotlib.pyplot as plt

EVAL_DEPTH = 1000          # ranking depth retrieved from every engine
BINARY_MIN_GRADE = 2       # 2 = strict relevance, 1 = lenient
RESULTS_TAG = f"stem{int(USE_STEMMING)}"     # keeps the stemming / no-stemming runs in separate files

def dcg(gains):
    gains = np.asarray(gains, dtype=np.float64)
    return float(np.sum((2.0 ** gains - 1.0) / np.log2(np.arange(2, len(gains) + 2)))) if len(gains) else 0.0

def ndcg_at_k(ranked, qrels, k):
    idcg = dcg(sorted(qrels.values(), reverse=True)[:k])
    if idcg == 0:
        return np.nan
    return dcg([qrels.get(int(i), 0) for i in ranked[:k]]) / idcg

def average_precision(ranked, rel_set, depth=100):
    """AP over the top `depth`, normalised by min(#relevant, depth)  (MAP@depth)."""
    if not rel_set:
        return np.nan
    hits, total = 0, 0.0
    for r, i in enumerate(ranked[:depth], 1):
        if int(i) in rel_set:
            hits += 1
            total += hits / r
    return total / min(len(rel_set), depth)

engines = {"BM25": bm25_engine, "Dense": dense_engine, "Hybrid": hybrid_engine}
hard_neg_sets = {q: set(a.tolist()) for q, a in hard_neg_dict.items()}
for e in engines.values():                      # warm-up so the first timed query is not penalised
    e.search("warm up query", top_k=10)

runs, latency_ms = {}, {n: [] for n in engines}
for spec in QUERY_SPECS:
    for name, eng in engines.items():
        t0 = time.perf_counter()
        idx, _ = eng.search(spec["query"], top_k=EVAL_DEPTH)
        latency_ms[name].append((time.perf_counter() - t0) * 1000.0)
        runs[(spec["query_id"], name)] = np.asarray(idx, dtype=np.int64)

rows = []
for spec in QUERY_SPECS:
    qid, qrels = spec["query_id"], qrels_dict[spec["query_id"]]
    rel = {i for i, g in qrels.items() if g >= BINARY_MIN_GRADE}
    for name in engines:
        r = runs[(qid, name)]
        rows.append(dict(
            query_id=qid, family=spec["family"], engine=name, n_relevant=len(rel),
            ndcg5=ndcg_at_k(r, qrels, 5), ndcg10=ndcg_at_k(r, qrels, 10),
            map100=average_precision(r, rel, 100),
            recall100=len(rel & set(map(int, r[:100]))) / len(rel) if rel else np.nan,
            recall1000=len(rel & set(map(int, r[:1000]))) / len(rel) if rel else np.nan,
            hardneg10=float(np.mean([int(i) in hard_neg_sets[qid] for i in r[:10]])) if len(r) else 0.0,
        ))
results_df = pd.DataFrame(rows)
METRICS = ["ndcg5", "ndcg10", "map100", "recall100", "recall1000", "hardneg10"]

summary_df = results_df.groupby("engine")[METRICS].mean().loc[list(engines)].round(4)
family_df = results_df.pivot_table(index="family", columns="engine", values="ndcg10", aggfunc="mean")[list(engines)].round(4)

bm25_mb = (sum(a.nbytes + t.nbytes for a, t in bm25_engine.inverted_index.values()) + bm25_engine.doc_len.nbytes) / 2**20
dense_mb = corpus_embeddings.nbytes / 2**20           # FAISS IndexFlatIP stores the same float32 matrix
resources_df = pd.DataFrame({
    "build_time_s": [getattr(bm25_engine, "fit_seconds", np.nan), globals().get("dense_encode_seconds") or np.nan, np.nan],
    "index_memory_MB": [bm25_mb, dense_mb, bm25_mb + dense_mb],
    "mean_query_ms": [float(np.mean(latency_ms[n])) for n in engines],
}, index=list(engines)).round(2)

print(f"Device: {device.upper()} | stemming: {USE_STEMMING} | binary relevance: grade >= {BINARY_MIN_GRADE} | depth: {EVAL_DEPTH}\n")
print("Mean over the 20 queries (hardneg10 = share of the top-10 that are hard negatives; lower is better):")
display(summary_df)
print("NDCG@10 by query family:")
display(family_df)
print("Resources (BM25 memory = postings arrays only; query time includes query encoding for Dense/Hybrid):")
display(resources_df)

results_df.to_csv(f"results_per_query_{RESULTS_TAG}.csv", index=False)
summary_df.to_csv(f"results_summary_{RESULTS_TAG}.csv")
family_df.to_csv(f"results_by_family_{RESULTS_TAG}.csv")
resources_df.to_csv(f"results_resources_{RESULTS_TAG}.csv")

In [ ]:
# ------------------------------------------------------------------
# Evaluation 2/2: the core experiment from the proposal.
# Coverage(q, d) = |Terms(q) ∩ Terms(d)| / |Terms(q)|   (stemmed, stop-word-free, negation words excluded, as in the qrels cell).
# For relevant (grade >= BINARY_MIN_GRADE) query-document pairs we measure the probability that each engine retrieves the document
# in its top-k, per coverage bucket. At most COVERAGE_SAMPLE_PER_QUERY relevant documents per query are used so that queries with
# thousands of relevant documents do not dominate. This is an observational stratification, not a causal experiment.
# ------------------------------------------------------------------
COVERAGE_BINS = [("high", 0.67, 1.01), ("medium", 0.34, 0.67), ("low", 0.10, 0.34), ("near-zero", 0.0, 0.10)]
COVERAGE_SAMPLE_PER_QUERY = 200
KS = (100, 1000)

def bucket_of(c):
    return next(name for name, lo, hi in COVERAGE_BINS if lo <= c < hi)

cov_rng = np.random.RandomState(SEED)
cov_rows = []
for spec in QUERY_SPECS:
    qid = spec["query_id"]
    q_terms = set(custom_tokenize(spec["query"])) - neg_tokens
    rel_idx = np.array([i for i, g in qrels_dict[qid].items() if g >= BINARY_MIN_GRADE], dtype=np.int64)
    if not q_terms or len(rel_idx) == 0:
        continue
    if len(rel_idx) > COVERAGE_SAMPLE_PER_QUERY:
        rel_idx = cov_rng.choice(rel_idx, COVERAGE_SAMPLE_PER_QUERY, replace=False)
    tops = {n: {k: set(map(int, runs[(qid, n)][:k])) for k in KS} for n in engines}
    for i in rel_idx:
        cov = len(q_terms & set(custom_tokenize(df.at[int(i), "search_text"]))) / len(q_terms)
        row = dict(query_id=qid, family=spec["family"], doc=int(i), coverage=cov, bucket=bucket_of(cov))
        for n in engines:
            for k in KS:
                row[f"{n}@{k}"] = int(int(i) in tops[n][k])
        cov_rows.append(row)
cov_df = pd.DataFrame(cov_rows)

rate_cols = [f"{n}@{k}" for k in KS for n in engines]
order = [b for b, _, _ in COVERAGE_BINS if b in set(cov_df["bucket"])]
coverage_table = cov_df.groupby("bucket")[rate_cols].mean().loc[order].round(3)
coverage_table.insert(0, "n_pairs", cov_df.groupby("bucket").size().loc[order])
print("P(relevant document retrieved in top-k) by coverage bucket (n_pairs = judged relevant pairs in the bucket):")
display(coverage_table)

fig, axes = plt.subplots(1, len(KS), figsize=(10, 3.6), sharey=True)
for ax, k in zip(np.atleast_1d(axes), KS):
    x, w = np.arange(len(coverage_table)), 0.26
    for j, n in enumerate(engines):
        ax.bar(x + (j - 1) * w, coverage_table[f"{n}@{k}"].values, w, label=n)
    ax.set_xticks(x)
    ax.set_xticklabels([f"{b}\n(n={int(m)})" for b, m in zip(coverage_table.index, coverage_table["n_pairs"])], fontsize=8)
    ax.set_title(f"Relevant doc retrieved in top-{k}, by query-document coverage", fontsize=9)
np.atleast_1d(axes)[0].set_ylabel("conditional recall")
np.atleast_1d(axes)[0].set_ylim(0, 1.2)
np.atleast_1d(axes)[0].legend(fontsize=8, loc="upper center", ncol=3)
plt.tight_layout()
plt.savefig(f"coverage_vs_recall_{RESULTS_TAG}.png", dpi=200)
plt.show()

# Complementarity at depth 100: which engine finds the relevant documents the other one misses?
b_hit, d_hit = cov_df["BM25@100"].astype(bool), cov_df["Dense@100"].astype(bool)
status = np.select([b_hit & d_hit, b_hit & ~d_hit, ~b_hit & d_hit], ["both", "BM25 only", "Dense only"], "neither")
comp_table = pd.crosstab(cov_df["family"], status)
comp_table = comp_table.reindex(columns=["both", "BM25 only", "Dense only", "neither"], fill_value=0)
comp_table.columns.name = "found by"
print("Relevant (sampled) documents found in the top-100 by BM25 / Dense / both / neither:")
display(comp_table)

cov_df.to_csv(f"coverage_pairs_{RESULTS_TAG}.csv", index=False)
coverage_table.to_csv(f"coverage_table_{RESULTS_TAG}.csv")
comp_table.to_csv(f"complementarity_{RESULTS_TAG}.csv")